# Module 7.1 — System Architecture & Orchestration

**Goal:** Run the full DeskMate pipeline — input guardrail → encoder triage → retriever → RAG decoder → output guardrails — as one orchestrated service. Verify the low-confidence short-circuit, measure per-stage latency, and exercise edge cases.

## 1. Setup

In [ ]:
# !pip install -q transformers sentence-transformers faiss-cpu rouge-score

import os, re, time, json, logging
import numpy as np
import matplotlib.pyplot as plt

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(message)s')
logger = logging.getLogger('deskmate')

SMOKE_TEST = True   # set False to load real models
CONFIDENCE_THRESHOLD = 0.70
MAX_TICKET_CHARS     = 2000
MIN_REPLY_TOKENS     = 20
FAITHFULNESS_GATE    = 0.60

os.makedirs('reports', exist_ok=True)
print('Config OK')

## 2. Evaluation Set

In [ ]:
GOLD = [
    {'ticket': 'I was charged twice for my subscription last month.',
     'intent': 'billing_dispute', 'confidence': 0.95,
     'ref': 'Contact support with your invoice numbers. We investigate and refund within 3 business days.'},
    {'ticket': 'How do I reset my two-factor authentication device?',
     'intent': 'account_access', 'confidence': 0.91,
     'ref': 'Go to Account > Security > Two-Factor Authentication and click Reset Device.'},
    {'ticket': 'The CSV export button is not working.',
     'intent': 'technical_bug', 'confidence': 0.88,
     'ref': 'Known issue ERR-500 fixed in version 4.3.0. Please update or contact support.'},
    {'ticket': 'Can I get a refund after 30 days?',
     'intent': 'billing_dispute', 'confidence': 0.82,
     'ref': 'DeskMate offers a 30-day money-back guarantee. Refunds after 30 days are evaluated case by case.'},
    {'ticket': 'My iOS app crashes on iPhone 14.',
     'intent': 'technical_bug', 'confidence': 0.79,
     'ref': 'A crash affecting iOS 17 was fixed in app version 2.1.3. Please update from the App Store.'},
]

EDGE_CASES = [
    {'ticket': '',         'label': 'empty_ticket'},
    {'ticket': 'X' * 2500, 'label': 'too_long'},
    {'ticket': 'Mi cuenta fue cobrada dos veces.', 'label': 'non_english'},
    {'ticket': 'What is the capital of France?',  'label': 'off_topic'},
    {'ticket': 'I bought your product.',           'label': 'low_confidence'},
]
print(f'Gold: {len(GOLD)}  Edge cases: {len(EDGE_CASES)}')

## 3. Pipeline Components (Stubs for Smoke Test)

In [ ]:
# These stubs replace the real models in SMOKE_TEST mode.
# In a full run, import from Modules 2.5, 4.3, 4.4, and the vLLM client.

def encoder_predict(ticket: str):
    if not ticket.strip() or len(ticket) > MAX_TICKET_CHARS:
        return 'unknown', 'unknown', 0.30
    if 'charged' in ticket.lower() or 'refund' in ticket.lower():
        return 'billing_dispute', 'DeskMate Pro', 0.92
    if 'csv' in ticket.lower() or 'crash' in ticket.lower() or 'export' in ticket.lower():
        return 'technical_bug', 'DeskMate Pro', 0.86
    if '2fa' in ticket.lower() or 'reset' in ticket.lower() or 'authentication' in ticket.lower():
        return 'account_access', 'DeskMate', 0.93
    if 'france' in ticket.lower() or 'capital' in ticket.lower():
        return 'general_inquiry', None, 0.45   # low confidence, off-topic
    return 'general_inquiry', 'DeskMate', 0.55  # low confidence

def full_retrieve(ticket: str, fields: dict = None):
    product = (fields or {}).get('product', '')
    return [
        {'id': 'c1', 'text': 'DeskMate offers a 30-day money-back guarantee.',
         'source': 'billing_refunds.md', 'section': 'refunds'},
        {'id': 'c2', 'text': 'For billing disputes, contact support with invoice numbers.',
         'source': 'billing_refunds.md', 'section': 'disputes'},
        {'id': 'c3', 'text': 'Known issue ERR-500 with CSV export, fixed in v4.3.0.',
         'source': 'technical_bugs.md', 'section': 'csv_bug'},
    ]

def build_rag_prompt(ticket: str, chunks: list):
    ctx = '\n\n'.join(
        f"[Source {i+1}] {c['source']}\n{c['text']}" for i, c in enumerate(chunks))
    return [
        {'role': 'system', 'content':
         'You are DeskMate. Answer using ONLY the context. Cite with [Source N].'},
        {'role': 'user', 'content': f'Context:\n{ctx}\n\nTicket: {ticket}'},
    ]

def vllm_generate_stub(prompt: list) -> str:
    ticket = prompt[-1]['content'].split('Ticket:')[-1].strip()
    if 'charged' in ticket.lower():
        return 'If you were charged twice, please contact support with your invoice numbers. [Source 1] [Source 2]'
    if 'csv' in ticket.lower() or 'export' in ticket.lower():
        return 'This is a known issue fixed in version 4.3.0. [Source 3]'
    if '2fa' in ticket.lower() or 'reset' in ticket.lower():
        return 'Go to Account > Security > Two-Factor Authentication and click Reset Device. [Source 1]'
    return 'I do not have that information - please contact support@deskmate.com.'

def extract_citations(reply: str, n_sources: int) -> list:
    cited = set()
    for m in re.finditer(r'\[Source (\d+)\]', reply):
        n = int(m.group(1))
        if 1 <= n <= n_sources:
            cited.add(n)
    return sorted(cited)

def faithfulness_score(reply: str, chunks: list) -> float:
    sents = [s.strip() for s in re.split(r'(?<=[.!?])\s+', reply.strip()) if s.strip()]
    if not sents: return 1.0
    source_words = set(re.findall(r'\b\w{4,}\b',
                                   ' '.join(c['text'] for c in chunks).lower()))
    faithful = sum(
        len(set(re.findall(r'\b\w{4,}\b', s.lower())) & source_words)
        / max(1, len(set(re.findall(r'\b\w{4,}\b', s.lower())))) >= 0.60
        for s in sents)
    return round(faithful / len(sents), 3)

print('Components defined.')

## 4. Input Guardrail

In [ ]:
def validate_input(ticket: str) -> tuple:
    if not ticket.strip():
        return False, 'empty_ticket'
    if len(ticket) > MAX_TICKET_CHARS:
        return False, 'too_long'
    return True, 'ok'

# Test edge cases
for ec in EDGE_CASES:
    valid, reason = validate_input(ec['ticket'])
    print(f"  {ec['label']:15s}: valid={valid}, reason={reason}")

## 5. Encoder Triage & Low-Confidence Short-Circuit

In [ ]:
def triage_ticket(ticket: str) -> dict:
    intent, product, confidence = encoder_predict(ticket)
    if confidence < CONFIDENCE_THRESHOLD:
        return {
            'action': 'escalate',
            'reason': 'low_confidence',
            'intent_hint': intent,
            'confidence': confidence,
        }
    return {'action': 'continue', 'intent': intent,
            'product': product, 'confidence': confidence}

# Test: normal ticket
r = triage_ticket(GOLD[0]['ticket'])
print('Normal ticket:', r)

# Test: low-confidence ticket
r_low = triage_ticket('I bought your product.')
print('Low-confidence:', r_low)

# Test: off-topic
r_off = triage_ticket('What is the capital of France?')
print('Off-topic:', r_off)

## 6. Output Guardrails

In [ ]:
FALLBACK_REPLY = "I don't have that information - please contact support@deskmate.com."

def apply_output_guardrails(reply: str, chunks: list) -> dict:
    citations = extract_citations(reply, len(chunks))
    faith     = faithfulness_score(reply, chunks)
    n_tokens  = len(reply.split())

    flags = {
        'no_citation':  len(citations) == 0,
        'low_faith':    faith < FAITHFULNESS_GATE,
        'short_reply':  n_tokens < MIN_REPLY_TOKENS,
    }

    if flags['no_citation']:
        reply     = FALLBACK_REPLY
        citations = []
        faith     = 1.0

    return {'reply': reply, 'citations': citations,
            'faithfulness': faith, 'guardrail_flags': flags}

# Test: good reply
chunks = full_retrieve(GOLD[0]['ticket'])
good_reply = vllm_generate_stub(build_rag_prompt(GOLD[0]['ticket'], chunks))
print('Good reply guardrail result:')
print(apply_output_guardrails(good_reply, chunks))

# Test: no citation (guardrail fires)
bad_reply = 'Please contact our team for assistance with your billing issue.'
print('\nNo-citation guardrail result:')
print(apply_output_guardrails(bad_reply, chunks))

## 7. Full Orchestrator

In [ ]:
def handle_ticket(ticket: str) -> dict:
    t0 = time.perf_counter()
    trace = {'ticket': ticket[:60], 'stages': {}}

    # 1. Input guardrail
    valid, reason = validate_input(ticket)
    if not valid:
        return {'action': 'reject', 'reason': reason, 'reply': None,
                'latency_ms': round((time.perf_counter()-t0)*1000)}

    # 2. Encoder triage
    t1 = time.perf_counter()
    triage_result = triage_ticket(ticket)
    trace['stages']['triage_ms'] = round((time.perf_counter()-t1)*1000)

    if triage_result['action'] == 'escalate':
        return {**triage_result,
                'reply': None,
                'latency_ms': round((time.perf_counter()-t0)*1000),
                '_trace': trace}

    intent  = triage_result['intent']
    product = triage_result['product']

    # 3. Account-access shortcut (template, no LLM)
    if intent == 'account_access' and triage_result['confidence'] >= 0.90:
        return {'action': 'template_reply',
                'reply': 'To reset your password, click Forgot password on the login page.',
                'intent': intent, 'citations': [],
                'latency_ms': round((time.perf_counter()-t0)*1000)}

    # 4. Retrieve
    t2 = time.perf_counter()
    chunks = full_retrieve(ticket, fields={'product': product})
    trace['stages']['retrieval_ms'] = round((time.perf_counter()-t2)*1000)

    # 5. Generate
    t3 = time.perf_counter()
    prompt = build_rag_prompt(ticket, chunks)
    raw_reply = vllm_generate_stub(prompt)
    trace['stages']['generation_ms'] = round((time.perf_counter()-t3)*1000)

    # 6. Output guardrails
    result = apply_output_guardrails(raw_reply, chunks)

    if any(result['guardrail_flags'].values()):
        logger.warning('guardrail_flag %s', result['guardrail_flags'])

    return {
        'action': 'reply',
        'reply': result['reply'],
        'citations': result['citations'],
        'intent': intent,
        'faithfulness': result['faithfulness'],
        'guardrail_flags': result['guardrail_flags'],
        'latency_ms': round((time.perf_counter()-t0)*1000),
        '_trace': trace,
    }

# Test single ticket
result = handle_ticket(GOLD[0]['ticket'])
print('Reply:', result['reply'])
print('Intent:', result['intent'])
print('Citations:', result['citations'])
print('Faithfulness:', result['faithfulness'])
print('Latency:', result['latency_ms'], 'ms')

## 8. Batch Evaluation — 20 Gold Tickets

In [ ]:
from rouge_score import rouge_scorer as rs_module
_scorer = rs_module.RougeScorer(['rougeL'], use_stemmer=True)

results = [handle_ticket(ex['ticket']) for ex in GOLD * 4]  # 20 tickets

actions   = [r['action'] for r in results]
latencies = [r['latency_ms'] for r in results]
faithfulness_vals = [r.get('faithfulness', None) for r in results]

print(f'Total tickets: {len(results)}')
print(f'Actions: { {a: actions.count(a) for a in set(actions)} }')
print(f'Median latency: {sorted(latencies)[len(latencies)//2]} ms')
faith_vals = [f for f in faithfulness_vals if f is not None]
if faith_vals:
    print(f'Mean faithfulness: {round(sum(faith_vals)/len(faith_vals), 3)}')

# ROUGE-L
reply_results = [(r, ex) for r, ex in zip(results, GOLD * 4)
                 if r['action'] in ('reply', 'template_reply') and r.get('reply')]
if reply_results:
    rouges = [_scorer.score(ex['ref'], r['reply'])['rougeL'].fmeasure
              for r, ex in reply_results]
    print(f'Mean ROUGE-L: {round(sum(rouges)/len(rouges), 4)}')

## 9. Guardrail Firing Rate

In [ ]:
flag_names = ['no_citation', 'low_faith', 'short_reply']
flag_counts = {f: 0 for f in flag_names}

for r in results:
    for flag in flag_names:
        if r.get('guardrail_flags', {}).get(flag):
            flag_counts[flag] += 1

n_reply = sum(1 for r in results if r['action'] == 'reply')
print('Guardrail firing rates (out of reply actions):')
for flag, count in flag_counts.items():
    rate = round(count / max(1, n_reply) * 100, 1)
    print(f'  {flag:15s}: {count}/{n_reply} ({rate}%)')

## 10. Latency Breakdown

In [ ]:
# Collect stage latencies from traces
stage_latencies = {'triage': [], 'retrieval': [], 'generation': []}
for r in results:
    trace = r.get('_trace', {}).get('stages', {})
    if trace.get('triage_ms'):    stage_latencies['triage'].append(trace['triage_ms'])
    if trace.get('retrieval_ms'): stage_latencies['retrieval'].append(trace['retrieval_ms'])
    if trace.get('generation_ms'):stage_latencies['generation'].append(trace['generation_ms'])

# Simulated realistic values for illustration
sim_stages = {'Input guard': 0.2, 'Encoder': 18.0, 'Retriever': 45.0,
              'Decoder': 1350.0, 'Output guard': 3.0}

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.barh(list(sim_stages.keys()), list(sim_stages.values()), color='steelblue')
ax.set_xlabel('Median latency (ms)')
ax.set_title('Per-Stage Latency (illustrative)')
ax.set_xscale('log')

ax = axes[1]
ax.hist(latencies, bins=10, color='coral', edgecolor='white')
ax.set_xlabel('Total request latency (ms)')
ax.set_ylabel('Count')
ax.set_title('End-to-End Latency Distribution')

plt.tight_layout()
plt.savefig('orchestration_latency.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: orchestration_latency.png')

## 11. Edge Case Tests

In [ ]:
print('Edge case results:\n')
for ec in EDGE_CASES:
    r = handle_ticket(ec['ticket'])
    action = r.get('action', 'unknown')
    reason = r.get('reason', r.get('intent_hint', ''))
    print(f"  {ec['label']:15s}: action={action}, reason/intent={reason}")

## 12. Checkpoint

In [ ]:
checkpoint = '''
Where does a low-confidence triage short-circuit the pipeline?

After the encoder step, before the retriever is called.

The encoder classifies intent and returns a confidence score (softmax probability).
If confidence < CONFIDENCE_THRESHOLD (0.70), the orchestrator immediately returns
an escalation response — no retrieval, no LLM call.

Why here?
  1. The decoder is the bottleneck in cost and latency. Short-circuiting before it
     saves ~99% of the per-request cost for ambiguous tickets.
  2. The encoder's best-guess intent is still returned as a routing hint for the
     human agent who handles the escalation.
  3. A low-confidence classification usually signals an ambiguous or off-topic
     ticket — exactly the cases where an LLM reply is most likely to be wrong.
'''
print(checkpoint)

## 13. Save Report

In [ ]:
action_counts = {a: actions.count(a) for a in set(actions)}
report = [
    '# Orchestration Report\n',
    f'Total tickets evaluated: {len(results)}',
    f'Smoke test: {SMOKE_TEST}\n',
    '## Action Distribution',
    '',
]
for action, count in action_counts.items():
    report.append(f'  {action}: {count}')
report += [
    '',
    '## Guardrail Firing Rates',
    '',
]
for flag, count in flag_counts.items():
    rate = round(count / max(1, n_reply) * 100, 1)
    report.append(f'  {flag}: {rate}%')
report += [
    '',
    f'Median end-to-end latency: {sorted(latencies)[len(latencies)//2]} ms',
    '',
    '## Checkpoint',
    '',
    'Short-circuit fires after encoder, before retriever.',
    'Confidence < 0.70 → escalate to human; no LLM call made.',
    'Saves ~99% of cost for ambiguous/off-topic tickets.',
]

with open('reports/orchestration_report.md', 'w') as f:
    f.write('\n'.join(report))

print('Saved: reports/orchestration_report.md')
print('\n=== Module 7.1 Complete ===')